## Imports das Bibliotecas

In [ ]:
## Imports Geral
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import nltk
import numpy as np

# Import para Visualização de Dados
from wordcloud import WordCloud

# Extração de Texto
from nltk.corpus import stopwords

# Importações de Metrica
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Preprocessamento
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

# Modelos
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import warnings

# Suprimir todos os warnings
warnings.filterwarnings("ignore")

# DESENVOLVER O MODELO SEM VIES 

# Analise dos dados

In [ ]:
df = pd.read_csv("cor_da_justica.csv")
remover_colunas = ["Unnamed: 0", "uris.uriAutores", "uris.uriPropPrincipal",
                   "uris.uriUltimoRelator", "foto", "cpf", ]
df = df.drop(columns=remover_colunas)
df = df.dropna(subset=["ementa"])
df = df.drop_duplicates()
df.info()

In [ ]:
df["ementa"] = df.apply(lambda x: x["ementa"] + ' ' +  x["ementaDetalhada"] if x["ementaDetalhada"] == str(x["ementaDetalhada"]) else x["ementa"] , axis=1)
df["ementa"] = df.apply(lambda x: x["ementa"] + ' ' +  x["ementa_principal"] if x["ementa_principal"] != str("vazio") else x["ementa"] , axis=1)
df["ementa"] = df.apply(lambda x: x["ementa"] + ' ' +  x["keywords"] if x["keywords"] == str(x["keywords"]) else x["ementa"] , axis=1)
df["ementa"] = df.apply(lambda x: x["ementa"] + ' ' +  x["keywords_principal"] if x["keywords_principal"] == str(x["keywords_principal"]) else x["ementa"] , axis=1)

In [ ]:
df = df.drop(columns=["ementaDetalhada", "ementa_principal", 'keywords', 'keywords_principal'])

In [ ]:
df["situacao"].loc[(df["situacao"] == "Aguardando Designação de Relator(a)" ) | (df["situacao"] == "Aguardando Parecer") | (df["situacao"] == "Aguardando Encaminhamento") | (df["situacao"] == "Aguardando Definição Encaminhamento") | (df["situacao"] == "Aguardando Recurso") | (df["situacao"] == "Aguardando Despacho de Arquivamento ") | (df["situacao"] == "Aguardando Deliberação")  | (df["situacao"] == "Aguardando Designação - Aguardando Devolução de Relator(a) que deixou de ser Membro") | (df["situacao"] == "Aguardando Análise de Parecer") | (df["situacao"] == "Arquivada") ] = "AGUARDANDO"
df["situacao"].loc[(df["situacao"] == "Tramitando em Conjunto" ) | (df["situacao"] == "Pronta para Pauta")] = "TRAMITANDO"
df["situacao"] = df["situacao"].fillna("RECUSADO")

In [ ]:
df = df.drop(df[(df['cor'] == "#NE") | (
    df["cor"] == "#NE#") | (df["cor"] == "NÃO DIVULGÁVEL")].index).reset_index(drop=True)

# df["cor"].loc[(df["cor"] == "AMARELA") | (
#     df["cor"] == "NÃO DIVULGÁVEL") | (df["cor"] == "INDÍGENA") | (df["cor"] == "PARDA") | (df["cor"] == "PRETA")] = "OUTROS"

In [ ]:
df['cor'].value_counts()

In [ ]:

## MODELOS SEM VIES

#Outros 100
#Brancos 100

#Gerar um classificar aleátorio / classificador sem viés tem que dar proximo a 50% de acurácia
# E macro F1 fica pior que 50%

#y_true = dados normais
#y_pred = se o valor for maior que 0.7 

#  f1_score(y_true, y_pred, average='macro')

branca = df["cor"].loc[(df["cor"] == 'BRANCA')].value_counts()
parda = df["cor"].loc[(df["cor"] == 'PARDA')].value_counts()
negra = df["cor"].loc[(df["cor"] == 'PRETA')].value_counts()
indigena = df["cor"].loc[(df["cor"] == 'INDÍGENA')].value_counts()
amarela = df["cor"].loc[(df["cor"] == 'AMARELA')].value_counts()

prob_brancos = (branca[0]/len(df["cor"]))
prob_pardas = (parda[0]/len(df["cor"]))
prob_negra = (negra[0]/len(df["cor"]))
prob_indigena = (indigena[0]/len(df["cor"]))
prob_amarela = (amarela[0]/len(df["cor"]))


y_prob = {'brancos': prob_brancos, 'pardas': prob_pardas, 'negra': prob_negra, 'indigena': prob_indigena, 'amarela': prob_amarela}


###obs: Aplicar para todos os modelos

In [ ]:
y_prob

In [ ]:
df["cor"]

In [ ]:
df['cor'].value_counts()

In [ ]:
cor = df["cor"].value_counts().reset_index()
cor["cor"].unique()
cor

In [ ]:
fx, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.bar(cor["cor"], height=cor["count"])
plt.show()

In [ ]:
df["ementa"].dropna().count()

## Bag Of Words

In [ ]:
# nltk.download('stopwords')

In [ ]:
df = df.dropna(subset=["ementa"]).reset_index(drop=True)

In [ ]:
tf_idf = TfidfVectorizer(stop_words=(stopwords.words('portuguese')), ngram_range=(1,1) )
X = tf_idf.fit_transform(df["ementa"])
vocab = tf_idf.get_feature_names_out()
vocab = ' '.join(vocab)


## WORD CLOUD

In [ ]:
le = LabelEncoder()

In [ ]:
# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    colormap='gist_rainbow',
    min_font_size=10).generate(vocab)

plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

In [ ]:
df_brancos = df[df["cor"] == "BRANCA"]
df_pardas  = df[df["cor"] == "PARDA"]
df_indigenas  = df[df["cor"] == "INDÍGENA"]
df_pretas  = df[df["cor"] == "PRETA"]
df_amarelas  = df[df["cor"] == "AMARELA"]



tf_idf_brancos = TfidfVectorizer(stop_words=(stopwords.words('portuguese')), ngram_range=(1,1) )
tf_idf_pardas = TfidfVectorizer(stop_words=(stopwords.words('portuguese')), ngram_range=(1,1) )


y_brancos = le.fit_transform(df_brancos["situacao"])
y_pardas = le.fit_transform(df_pardas["situacao"])
y_indigena = le.fit_transform(df_indigenas["situacao"])
y_preta = le.fit_transform(df_pretas["situacao"])
y_amarela = le.fit_transform(df_amarelas["situacao"])

In [ ]:
X_brancos = tf_idf_brancos.fit_transform(df_brancos["ementa"])
vocab_brancos = tf_idf_brancos.get_feature_names_out()
vocab_brancos = ' '.join(vocab_brancos)


In [ ]:
vocab_brancos

In [ ]:
X_pardas = tf_idf.fit_transform(df_pardas["ementa"])
vocab_pardas = tf_idf.get_feature_names_out()
vocab_pardas = ' '.join(vocab_pardas)


In [ ]:
X_indigena = tf_idf.fit_transform(df_indigenas["ementa"])
vocab_indigena = tf_idf.get_feature_names_out()
vocab_indigena = ' '.join(vocab_indigena)


In [ ]:
X_pretas = tf_idf.fit_transform(df_pretas["ementa"])
vocab_pretas = tf_idf.get_feature_names_out()
vocab_pretas = ' '.join(vocab_pretas)


In [ ]:
X_amarelas = tf_idf.fit_transform(df_amarelas["ementa"])
vocab_amarelas = tf_idf.get_feature_names_out()
vocab_amarelas = ' '.join(vocab_amarelas)


## GRUPO 1: PESSOA DE COR DE PELE BRANCA

In [ ]:

# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    min_font_size=10).generate(vocab_brancos)
plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.title('Brancos')
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

## GRUPO 2: PESSOA DE COR DE PELE DIFERENTES 

# PARDAS

In [ ]:

# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    colormap='flag_r',
    min_font_size=10).generate(vocab_pardas)
plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.title('Pardos')
plt.show()

# NEGRAS

In [ ]:

# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    colormap='cool',
    min_font_size=10).generate(vocab_pretas)
plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.title('Pretas')
plt.show()

# INDÍGENAS

In [ ]:

# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    colormap='inferno',
    min_font_size=10).generate(vocab_indigena)
plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.title('Indígenas')
plt.show()

# AMARELAS

In [ ]:

# Juntar as strings em uma única string separada por espaços
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',
    colormap='winter',
    min_font_size=10).generate(vocab_amarelas)
plt.figure(figsize=(10, 6), facecolor=None)
plt.imshow(wordcloud)
plt.axis('off')
plt.tight_layout(pad=0)
plt.title('Amarelas')
plt.show()

# PCA

In [ ]:
df['cor']

In [ ]:
le = LabelEncoder()

df["cor"] = le.fit_transform(df["cor"])
y = df["cor"]

In [ ]:
y

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.toarray())
df_pca = pd.DataFrame(data=X_pca, columns=['ementa', 'situacao'])
df_pca["target"] = df["cor"].reset_index(drop=True)


sns.scatterplot(x='ementa', y="situacao", data=df_pca)
plt.title('PCA - Não Rotulado')
plt.show()

In [ ]:
y_map = {0: 'Amarela', 1: 'Branca', 2: 'Indigena', 3: 'Parda', 4: 'Negra'}

df_pca["target"] = df["cor"].map(y_map)

sns.scatterplot(x='ementa', y="situacao", hue="target", data=df_pca)
plt.title('PCA - Rotulado')
plt.show()

## Treino do Modelo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from imblearn.over_sampling import SMOTE

# transform the dataset
oversample = SMOTE()
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [ ]:
## Funçao para Avaliação dos Modelos

def evaluation_parametrics(name, y_test, y_pred):
    cm_test = confusion_matrix(y_test, y_pred)
    
    t1 = ConfusionMatrixDisplay(cm_test, display_labels=[
                                "Amarela","Branca", "Indígena", 'Parda', 'Negra'])
    print("Classification Report for Data Test")
    print(classification_report(y_test, y_pred))
    t1.plot()

In [ ]:
#K Fold

K = 5
kf = KFold(n_splits=K, shuffle=False) 

# Logistic Regression

In [ ]:
model_LR = LogisticRegression(random_state=42, max_iter=1000, solver="liblinear", penalty="l2", C=3, tol=0.001)
cross_val_results = cross_val_score(model_LR, X, y, cv=kf, scoring="f1_macro")
cross_val_results_accuracy = cross_val_score(model_LR, X, y, cv=kf, scoring="accuracy")

print("Resultado da Cross Validation: ", cross_val_results)


print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_LR.fit(X_train, y_train)
y_pred = model_LR.predict(X_test)
evaluation_parametrics("Machine Learning - Logistic", y_test, y_pred)

# SGD Classifier

In [ ]:
model_SGD = SGDClassifier(random_state=42, loss="squared_hinge", alpha=0.0001, learning_rate="constant", eta0=0.1, power_t=0.5, penalty="l2", tol=0.001, early_stopping=True, n_jobs=-1, class_weight="balanced")
cross_val_results = cross_val_score(model_SGD, X, y, cv=kf, scoring="f1_macro")
cross_val_results_accuracy = cross_val_score(model_SGD, X, y, cv=kf, scoring="accuracy")

print("Resultado da Cross Validation: ", cross_val_results)

print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_SGD.fit(X_train, y_train)
y_pred = model_SGD.predict(X_test)
evaluation_parametrics("Machine Learning - SGD", y_test, y_pred)

# K Neighbors Classifier

In [ ]:
model_KNeighbors = KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="auto", leaf_size=50, p=2 )
cross_val_results = cross_val_score(model_KNeighbors, X, y, cv=kf, scoring="f1_macro", )
cross_val_results_accuracy = cross_val_score(model_KNeighbors, X, y, cv=kf, scoring="accuracy")

print("Resultado da Cross Validation: ", cross_val_results)

print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_KNeighbors.fit(X_train, y_train)
y_pred = model_KNeighbors.predict(X_test)
evaluation_parametrics("Machine Learning - K Neighbors", y_test, y_pred)

# SVM (SUPPORT VECTOR MACHINE)

In [ ]:
model_SVM = LinearSVC(random_state=42, loss='squared_hinge', max_iter=1000, penalty="l2", C=3, tol=0.0001, multi_class='ovr', dual=True, class_weight="balanced",  )
cross_val_results = cross_val_score(model_SVM, X, y, cv=kf, scoring="f1_macro")
cross_val_results_accuracy = cross_val_score(model_SVM, X, y, cv=kf, scoring="accuracy")

print("Resultado da Cross Validation: ", cross_val_results)

print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_SVM.fit(X_train, y_train)
y_pred = model_SVM.predict(X_test)
evaluation_parametrics("Machine Learning - SVM", y_test, y_pred)

# Random Forest

In [ ]:
model_Random_Forest = RandomForestClassifier(random_state=42, n_estimators=500, criterion="gini", max_features='sqrt', class_weight='balanced', n_jobs=-1, )
cross_val_results = cross_val_score(model_Random_Forest, X, y, cv=kf, scoring="f1_macro")
cross_val_results_accuracy = cross_val_score(model_Random_Forest, X, y, cv=kf, scoring="accuracy")


print("Resultado da Cross Validation: ", cross_val_results)

print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_Random_Forest.fit(X_train, y_train)
y_pred = model_Random_Forest.predict(X_test)
evaluation_parametrics("Machine Learning - Random Forest", y_test, y_pred)

# Decision Tree

In [ ]:
model_Decision_Tree = DecisionTreeClassifier(random_state=42, criterion="entropy", splitter="random", class_weight="balanced", ccp_alpha=0.0001)

cross_val_results = cross_val_score(model_Decision_Tree, X, y, cv=kf, scoring="f1_macro")
cross_val_results_accuracy = cross_val_score(model_Decision_Tree, X, y, cv=kf, scoring="accuracy")

print("Resultado da Cross Validation: ", cross_val_results)

print("Media da Accuracy: ", cross_val_results_accuracy.mean())
print("Media da MACRO F1: ", cross_val_results.mean())
print("Desvio Padrão da MACRO F1: ", cross_val_results.std())

In [ ]:
model_Decision_Tree.fit(X_train, y_train)
y_pred = model_Decision_Tree.predict(X_test)
evaluation_parametrics("Machine Learning - Decision Tree", y_test, y_pred)

In [ ]:
## Funçao para Avaliação dos Modelos
# Refazer essa parte 
import numpy as np

amarelos = df["cor"].loc[(df["cor"] == 0)].value_counts()
prob_amarelos = (amarelos/len(df["cor"]))

branca = df["cor"].loc[(df["cor"] == 1)].value_counts()
prob_brancos = (branca/len(df["cor"]))

indigena = df["cor"].loc[(df["cor"] == 2)].value_counts()
prob_indigena = (indigena/len(df["cor"]))

pardos = df["cor"].loc[(df["cor"] == 3)].value_counts()
prob_pardos = (pardos/len(df["cor"]))

pretos = df["cor"].loc[(df["cor"] == 4)].value_counts()
prob_pretos = (pretos/len(df["cor"]))


print(prob_amarelos)
print(prob_brancos)
print(prob_indigena)
print(prob_pardos)
print(prob_pretos)

rng = np.random.default_rng(62)
numbers = rng.random(len(y_test))



for i in numbers:

    if i < prob_brancos:
        aux = df.loc[df["cor"] == 1].samples(1)

    elif i > prob_brancos and i < prob_branco + prob_pardos:
        aux = df.loc[df["cor"] == 3].samples(1)

    elif i > prob_branco + prob_pardos and i < prob_branco + prob_pardos + prob_pretos:
        aux = df.loc[df["cor"] == 3].samples(1)

    elif i > prob_branco + prob_pardos + prob_pretos  and i < prob_branco + prob_pardos + prob_pretos + prob_amarelos:
        aux = df.loc[df["cor"] == 3].samples(1)

    elif i > prob_branco + prob_pardos + prob_pretos + prob_amarelos:
        aux = df.loc[df["cor"] == 3].samples(1)

# balanceamento = [df.loc[df["cor"] == 0].sample(1)  if i < prob_brancos else df.loc[df['cor'] == 1].sample(1) for i in numbers]
# df = pd.concat(balanceamento)


# from sklearn.metrics import f1_score

# f1_score(y_test, df['cor'], average='macro')

In [ ]:
df["cor"].value_counts()